# CRA Tutorial — Hands-on AI on the National Research Platform

A short, self-contained tour of running AI on NRP, entirely **inside this
JupyterHub session** — no separate pods to launch and no YAML to apply.

A one-line vocabulary primer, in case these are new:

- **LLM** — a large language model (the thing that answers prompts).
- **Inference** — running a prompt through a model to get an answer.
- **RAG** — *retrieval-augmented generation*: look up relevant text first,
  then ask the model to answer **using only that text**. It's how you get an
  LLM to answer questions about your own docs instead of guessing.

**What you'll do**

1. Verify the session has the env vars, `kubectl`, and (optionally) a GPU.
2. Call the **NRP managed LLM** (`https://ellm.nrp-nautilus.io/v1`) from Python.
3. Run a **local LLM** (Ollama + `mistral`) on the session's GPU and ask the
   same prompt — managed vs local, side by side.
4. Build a **simple RAG pipeline** over a handful of facts using the
   NRP-managed Milvus, then re-ask the question through both backends.

When you're done, the README has a short **`opencode`** example that points an
agentic coding CLI at the same managed LLM.

**How to run a notebook (skip if you've used Jupyter before)**

- A notebook is a list of **cells**. Grey cells are code; this one is text.
- Click a cell and press **Shift + Enter** to run it and move on (or click ▶).
  Run the cells **top to bottom, in order** — later cells reuse earlier ones.
- A `[*]` means a cell is still running; a number like `[7]` means it's done.
  If something gets stuck, use **Kernel → Restart Kernel** and re-run from top.
- Cells that print `Skipping — no GPU` are doing the right thing on a
  CPU-only session; just keep going.

**Prerequisites**

- Spawn this session with **1 × NVIDIA-A10 GPU** only if you want to run the
  local-Ollama half of section 3. The managed-LLM and RAG sections work fine
  on a CPU-only session.
- The session pod already exports `OPENAI_API_BASE` and `OPENAI_API_KEY` and
  has `kubectl` wired to the `nrp-training-k8s` namespace.


## 1. Setup check

Confirm the session has everything we'll need: the managed-LLM env vars,
`kubectl` against `nrp-training-k8s`, and `nvidia-smi` if you spawned with a
GPU. The Ollama sections will skip themselves gracefully if there's no GPU.


In [ ]:
import os, shutil, subprocess

print("OPENAI_API_BASE =", os.environ.get("OPENAI_API_BASE", "NOT SET"))
print("OPENAI_API_KEY  =", "set (" + str(len(os.environ.get("OPENAI_API_KEY",""))) + " chars)"
      if os.environ.get("OPENAI_API_KEY") else "NOT SET")

kubectl = shutil.which("kubectl") or "/opt/conda/bin/kubectl"
print("kubectl path    =", kubectl)
print("kubectl version =", subprocess.run([kubectl, "version", "--client", "-o", "json"],
                                          capture_output=True, text=True).stdout[:120], "...")

print("can list pods in nrp-training-k8s:",
      subprocess.run([kubectl, "auth", "can-i", "list", "pods", "-n", "nrp-training-k8s"],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
# GPU detection. Section 3 (local Ollama) needs this; others are fine without.
import subprocess

HAS_GPU = False
try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True).stdout.strip()
    print("GPU(s) detected:")
    print(out)
    HAS_GPU = True
except (FileNotFoundError, subprocess.CalledProcessError):
    print("No GPU in this session pod.")
    print("Section 3 (local Ollama) will skip the local-LLM half.")
    print("To run the full comparison, respawn with 1 x NVIDIA-A10 from the JupyterHub launcher.")


## 2. NRP managed LLM

NRP hosts a rotating catalog of open-weights LLMs behind an
OpenAI-compatible URL: `https://ellm.nrp-nautilus.io/v1`. You don't run a
pod, you don't pay GPU time — you just send HTTP requests with the bearer
token already exported as `OPENAI_API_KEY`.

The same `openai` Python SDK code targets:

- the **NRP managed endpoint** (this section),
- a **local Ollama / vLLM / TGI server** (section 3),
- or the OpenAI cloud — just by changing `base_url`.

This portability is the entire point of OpenAI-compatible APIs.


### What the next five cells do

Each one does a single small thing, so you can see every moving part:

1. **List the catalog** — ask the endpoint which models it's serving right now.
2. **Define a `chat()` helper** — a tiny wrapper so every later call is one line.
3. **Ask one question** — a single request and response.
4. **Stream** — the same call, but printing tokens as they're generated.
5. **Swap the model** — identical code, just a different `model=` name.

Notice that the only NRP-specific thing is the `base_url`. Everything else is
the standard `openai` library you'd use against OpenAI itself.


In [ ]:
# List the models currently served by the NRP managed endpoint.
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
models = client.models.list()
for m in models.data:
    print(f"  {m.id:<20}  (owned_by={getattr(m, 'owned_by', '?')})")


In [ ]:
# A tiny helper we'll reuse across this notebook. It bounds `max_tokens` so a
# runaway response can't hang the kernel, and falls back to a string if a
# model puts its reply in `reasoning` instead of `content` (some reasoning
# models on this endpoint do that).
def chat(prompt, model="gemma", system=None, max_tokens=300, llm=None):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = (llm or client).chat.completions.create(
        model=model, messages=msgs, max_tokens=max_tokens, temperature=0.2,
    )
    return resp.choices[0].message.content or "(model produced no `content` field — try a different model or raise max_tokens)"


In [ ]:
# Single chat completion.
print(chat(
    "What is the National Research Platform?",
    system="Answer in one sentence.",
))


In [ ]:
# Streaming — tokens arrive as they're generated.
stream = client.chat.completions.create(
    model="gemma",
    messages=[{"role": "user", "content": "Write a haiku about GPUs."}],
    stream=True,
    max_tokens=80,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


In [ ]:
# Switch to a smaller model. Same code, just change the `model` arg.
print(chat(
    "Explain Kubernetes namespaces in two sentences.",
    model="gemma-small",
))

# Try the catalog's reasoning-capable models too. Some put their reply in
# `reasoning` and leave `content` null; the helper above falls back gracefully
# so the cell doesn't crash.
for m in ["minimax-m2", "qwen3", "gpt-oss"]:
    try:
        print(f"\n--- {m} ---")
        print(chat("Name one strength of this model in five words.", model=m, max_tokens=200))
    except Exception as e:
        print(f"  skipped ({type(e).__name__}: {str(e)[:80]})")


<details>
<summary><b>What this would look like as a Kubernetes pod (click to expand)</b></summary>

If you wanted to run your *own* LLM server on NRP instead of using the managed
endpoint, you'd request a GPU pod that boots an OpenAI-compatible server like
vLLM or HuggingFace TGI. The pod below would expose `/v1/chat/completions` on
the same port, so the Python code above works against it unchanged — only the
`base_url` changes (e.g., to `http://127.0.0.1:8080/v1` after a port-forward).

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: tutorial-<username>-tgi
  namespace: nrp-training-k8s
spec:
  containers:
  - name: tgi
    image: ghcr.io/huggingface/text-generation-inference:2.1.1
    args: ["--model-id", "HuggingFaceH4/zephyr-7b-beta"]
    resources:
      requests: { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
      limits:   { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
        - matchExpressions:
          - { key: nvidia.com/gpu.product, operator: In, values: [NVIDIA-A10] }
  tolerations:
  - { key: nautilus.io/reservation, operator: Equal, value: nrp, effect: NoSchedule }
```

Compared to the managed endpoint:

| | Managed (`ellm.nrp-nautilus.io`) | Self-hosted pod (YAML above) |
|---|---|---|
| Setup | none — just an API call | `kubectl apply` + wait for pull + port-forward |
| Cost | shared, no GPU billed to you | a GPU on the workshop reservation |
| Control | pick from the catalog | pick *any* HF model, quantization, runtime flags |
| Use when | classroom demos, prototypes | research that needs a specific model or config |

</details>


## 3. Local LLM in the notebook — Ollama on the session GPU

Same OpenAI-compatible API, but the model runs on **your** GPU inside this
JupyterHub session. We'll install Ollama, pull `mistral` (7B Q4, ~4 GB), and
hit it at `http://localhost:11434/v1` — then ask the same prompt against
NRP's managed `gemma` and our local `mistral` side by side.

> The cells below skip themselves if `HAS_GPU` is `False`. Respawn with 1 ×
> NVIDIA-A10 if you want to run the local half.


In [ ]:
# Install Ollama into the session pod. The official install script needs sudo,
# which is blocked by the pod's seccomp/noNewPrivileges policy, so we pull the
# release tarball directly and drop the binary in ~/.local/bin.
# Idempotent: re-running is a no-op once the binary exists.
import os, shutil, subprocess

OLLAMA_VERSION = "v0.24.0"  # pin to a known-good release

if not HAS_GPU:
    print("Skipping Ollama install — no GPU in this session.")
else:
    if shutil.which("ollama") or os.path.exists(os.path.expanduser("~/.local/bin/ollama")):
        print("Ollama already installed at", shutil.which("ollama") or "~/.local/bin/ollama")
    else:
        print(f"Downloading Ollama {OLLAMA_VERSION} (~80 MB, ~30 seconds)...")
        os.makedirs(os.path.expanduser("~/.local/bin"), exist_ok=True)
        os.makedirs(os.path.expanduser("~/.local/lib"), exist_ok=True)
        url = f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
        subprocess.run(
            f"curl -fsSL {url} | tar --use-compress-program=unzstd -xf - -C ~/.local",
            shell=True, check=True,
        )
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
    print("ollama binary at:", shutil.which("ollama"))


In [ ]:
# Start ollama serve in the background and wait until it responds.
# Notes for this pod:
#   - OLLAMA_GPU_DISCOVERY_TIMEOUT bumped from default (5s) — the JH session
#     can be CPU-starved, and the GPU-probe runner needs breathing room.
#   - We accept the bare TimeoutError that urllib raises on slow responses.
import os, socket, subprocess, time, urllib.request, urllib.error

def ollama_alive():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return True
    except (urllib.error.URLError, ConnectionResetError, OSError, socket.timeout, TimeoutError):
        return False

if not HAS_GPU:
    print("Skipping Ollama serve — no GPU in this session.")
elif ollama_alive():
    print("Ollama already running.")
else:
    print("Starting `ollama serve` in the background...")
    ollama_env = {
        **os.environ,
        "OLLAMA_HOST": "0.0.0.0:11434",
        "OLLAMA_GPU_DISCOVERY_TIMEOUT": "60",
        # CPU-starved JH pods can take >5min to warm the model the first time,
        # so we bump the load timeout. After the model is in VRAM, calls are fast.
        "OLLAMA_LOAD_TIMEOUT": "15m",
        # Ollama's bundled cuda libs live next to its binary
        "OLLAMA_LIBRARY_PATH": os.path.expanduser("~/.local/lib/ollama"),
    }
    subprocess.Popen(
        "nohup ollama serve > /tmp/ollama.log 2>&1 &",
        shell=True, env=ollama_env,
    )
    deadline = time.time() + 90
    while time.time() < deadline:
        time.sleep(2)
        if ollama_alive():
            print(f"Ollama up after ~{int(time.time() - (deadline - 90))}s.")
            break
    else:
        print("Ollama did not come up within 90s. Check /tmp/ollama.log")


In [ ]:
# Pull the mistral 7B model (~4 GB). First time only.
import subprocess, time, json, urllib.request

def already_pulled(name="mistral"):
    try:
        r = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2))
        return any(name in m["name"] for m in r.get("models", []))
    except Exception:
        return False

if not HAS_GPU:
    print("Skipping — no GPU.")
elif already_pulled("mistral"):
    print("mistral already pulled.")
else:
    print("Pulling mistral... (~4 GB, 2-4 minutes the first time)")
    # Drive the pull via the HTTP API instead of `ollama pull`, because the CLI
    # writes \r-based progress bars that buffer indefinitely under nbconvert.
    body = json.dumps({"name": "mistral", "stream": True}).encode()
    req  = urllib.request.Request("http://localhost:11434/api/pull", data=body,
                                  headers={"Content-Type": "application/json"})
    t0 = time.time()
    last_status = None
    with urllib.request.urlopen(req) as resp:
        for raw in resp:
            try:
                evt = json.loads(raw.decode())
            except Exception:
                continue
            status = evt.get("status", "")
            # Only print on status transitions (not every kilobyte) to keep output tidy
            if status and status != last_status:
                print(f"  [{int(time.time()-t0):3d}s] {status}")
                last_status = status
            if evt.get("error"):
                raise RuntimeError(evt["error"])
    print(f"Done in {int(time.time()-t0)}s.")


In [ ]:
# Hello-world against the local model. Same OpenAI client, different base_url.
# Heads up: the first call after `ollama serve` started will be slow (~30-120s
# while llama.cpp loads the 4 GB of weights into VRAM). Subsequent calls in
# this notebook reuse the loaded model and respond in 1-2 seconds.
from openai import OpenAI

if HAS_GPU:
    local = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1",
                   timeout=900)  # generous timeout for the cold load
    print(chat(
        "Say hi from the GPU in this JupyterHub session.",
        model="mistral", llm=local, max_tokens=80,
    ))
else:
    print("Skipping — no GPU.")


In [ ]:
# Side-by-side: same prompt, managed gemma vs local mistral.
prompt = "Explain in two sentences why a research cluster might use Kubernetes namespaces."

print("=" * 78)
print("NRP MANAGED (gemma)")
print("=" * 78)
print(chat(prompt, model="gemma"))

if HAS_GPU:
    print()
    print("=" * 78)
    print("LOCAL (mistral, on this pod's GPU)")
    print("=" * 78)
    print(chat(prompt, model="mistral", llm=local))
else:
    print("\n(Local half skipped — no GPU.)")


**What to notice.** The Python code is byte-for-byte identical — same SDK,
same `chat.completions.create`, same messages. Only `base_url` differs.
Latency, answer style, and request privacy will differ:

| | Managed `gemma` | Local `mistral` (this pod) |
|---|---|---|
| Where it runs | NRP GPUs you don't see | the A10 attached to this pod |
| Latency to first token | ~1.5 s | ~6 s (warm) |
| GPU billed | none | yours, while pod runs |
| Privacy | request leaves your namespace | never leaves your pod |
| Catalog | the NRP model list | anything `ollama pull` supports |

<details>
<summary><b>What the Ollama equivalent looks like as a YAML pod (click to expand)</b></summary>

If you ran Ollama in its own dedicated pod instead of inside the JupyterHub
session (e.g., to share one Ollama server across multiple notebook sessions,
or to keep model weights on a separate PVC), it would look like this:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: tutorial-<username>-ollama
  namespace: nrp-training-k8s
spec:
  containers:
  - name: ollama
    image: ollama/ollama:0.2.8
    resources:
      requests: { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
      limits:   { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
    ports:
    - { containerPort: 11434 }
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
        - matchExpressions:
          - { key: nvidia.com/gpu.product, operator: In, values: [NVIDIA-A10] }
  tolerations:
  - { key: nautilus.io/reservation, operator: Equal, value: nrp, effect: NoSchedule }
```

You'd then `kubectl port-forward pod/tutorial-<username>-ollama 11434:11434`
and point your notebook at `http://localhost:11434/v1` — same code, same
result. The in-notebook path you just ran is the same thing, minus the YAML.

</details>


## 4. Simple RAG with managed Milvus

RAG ("retrieval-augmented generation") is the standard pattern for asking an
LLM about facts the model wasn't trained on: you (1) embed your corpus into a
vector database, (2) retrieve the chunks closest to the user's question, and
(3) include those chunks in the prompt with a "answer only from this context"
system message.

NRP runs a managed [Milvus](https://milvus.io) at `milvus.nrp-nautilus.io:50051`.
The credentials live in a Kubernetes Secret in our namespace, which we'll
load via `kubectl` (one shell-out, then it's just Python).

This first RAG pass uses six hand-written facts so you can see every moving
part. The same pipeline scales to a real corpus.


### The five mechanical steps of RAG

The cells below walk through RAG one step at a time:

1. **Load credentials** for the vector database (Milvus) from a Kubernetes Secret.
2. **Create a collection** — a table that stores text plus its embedding vector.
3. **Embed + insert** your documents (turn each chunk of text into a vector).
4. **Retrieve** — embed the *question* and find the closest chunks.
5. **Ask** — hand those chunks to the LLM with "answer only from this context."

An **embedding** is just a list of numbers that captures a piece of text's
meaning; chunks about the same topic land near each other, which is what makes
"find the closest chunks" work.


In [ ]:
# Pull Milvus creds from the namespace Secret. The kubeconfig in this pod
# already points at the jupyterhub-sa service account which has read on this Secret.
import os, base64, subprocess

def k_secret(key):
    out = subprocess.check_output([
        "kubectl", "get", "secret", "nrp-training-milvus-credentials",
        "-n", "nrp-training-k8s", "-o", f"jsonpath={{.data.{key}}}",
    ])
    return base64.b64decode(out).decode()

os.environ["MILVUS_HOST"]     = k_secret("host")
os.environ["MILVUS_PORT"]     = k_secret("port")
os.environ["MILVUS_USER"]     = k_secret("username")
os.environ["MILVUS_PASSWORD"] = k_secret("password")
os.environ["MILVUS_SECURE"]   = k_secret("secure")
os.environ["MILVUS_DB_NAME"]  = k_secret("database")

print(f"Milvus: {os.environ['MILVUS_USER']}@{os.environ['MILVUS_HOST']}:{os.environ['MILVUS_PORT']}")
print(f"  database = {os.environ['MILVUS_DB_NAME']}")
print(f"  secure   = {os.environ['MILVUS_SECURE']}")


In [ ]:
# Install RAG dependencies. The PyTorch image already has torch + transformers,
# so we just add pymilvus, sentence-transformers, and the text splitter.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymilvus>=2.3", "sentence-transformers>=2.2",
                "langchain-text-splitters>=0.0.1"], check=True)
print("Dependencies installed.")


In [ ]:
# Connect to Milvus and create a per-user collection for this demo.
import os, getpass, re
from pymilvus import (
    connections, utility, Collection, CollectionSchema, FieldSchema, DataType,
)

connections.connect(
    alias="default",
    host=os.environ["MILVUS_HOST"], port=os.environ["MILVUS_PORT"],
    user=os.environ["MILVUS_USER"], password=os.environ["MILVUS_PASSWORD"],
    secure=os.environ["MILVUS_SECURE"].lower() == "true",
    db_name=os.environ["MILVUS_DB_NAME"],
)

# Per-user collection so demos don't collide.
user_slug = re.sub(r"[^a-z0-9_]+", "_", getpass.getuser().lower())[:32]
SIMPLE_COLLECTION = f"simple_rag_demo_{user_slug}"

if utility.has_collection(SIMPLE_COLLECTION):
    print(f"Dropping existing collection {SIMPLE_COLLECTION} for a clean demo run...")
    utility.drop_collection(SIMPLE_COLLECTION)

schema = CollectionSchema(fields=[
    FieldSchema(name="pk",     dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=128),
    FieldSchema(name="text",   dtype=DataType.VARCHAR, max_length=4000),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=384),
])
coll = Collection(name=SIMPLE_COLLECTION, schema=schema,
                  description="AI Unlocked simple RAG demo corpus")
coll.create_index("vector", {"metric_type": "COSINE", "index_type": "AUTOINDEX", "params": {}})
print(f"Created collection: {SIMPLE_COLLECTION}")


In [ ]:
# Six hand-written facts about NRP. In a real workflow you'd point this at a
# document folder; we keep it tiny so you can read the whole corpus.
DOCS = [
    ("nrp-overview",
     "The National Research Platform (NRP) is a shared national cyberinfrastructure "
     "built on the Nautilus Kubernetes cluster. It provides hundreds of GPU nodes, "
     "shared storage, and managed services like JupyterHub and a managed LLM endpoint."),
    ("nrp-gpus",
     "NRP exposes GPUs through Kubernetes resource keys. Generic GPUs use "
     "nvidia.com/gpu, specific products use keys like nvidia.com/a100 or "
     "nvidia.com/h200. Pods can also constrain themselves to a model with "
     "nvidia.com/gpu.product node-affinity."),
    ("nrp-managed-llm",
     "NRP runs a managed LLM service at https://ellm.nrp-nautilus.io/v1. It is "
     "OpenAI-compatible. Users authenticate with a bearer token minted at "
     "https://nrp.ai/llmtoken. The catalog rotates and includes qwen3, gpt-oss, "
     "gemma, and minimax-m2 among others."),
    ("nrp-milvus",
     "NRP runs a managed Milvus vector database reachable at "
     "milvus.nrp-nautilus.io:50051 (gRPC, TLS on). Per-user credentials are "
     "minted at https://nrp.ai/milvus."),
    ("nrp-policy-limits",
     "All workloads on NRP must declare CPU and memory requests and limits. A "
     "Gatekeeper admission policy rejects pods that omit them. The pattern used "
     "in workshop manifests is requests == limits."),
    ("nrp-policy-sleep",
     "Pods that idle indefinitely (e.g., `sleep infinity` in a Job) are against "
     "NRP policy and will result in the user being banned from the cluster. "
     "Always run real workloads or batch jobs with a defined exit."),
]
print(f"Loaded {len(DOCS)} hand-written facts.")


In [ ]:
# Embed with all-MiniLM-L6-v2 (384 dim, runs fine on CPU) and insert.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
sources = [s for s, _ in DOCS]
texts   = [t for _, t in DOCS]
vectors = embedder.encode(texts, normalize_embeddings=True).tolist()

coll.insert([sources, texts, vectors])
coll.flush()
coll.load()
print(f"Inserted {len(DOCS)} chunks; collection now has {coll.num_entities} entities.")


In [ ]:
# Helper: retrieve top-k chunks for a question.
def retrieve(question, k=3):
    qvec = embedder.encode([question], normalize_embeddings=True).tolist()
    hits = coll.search(
        data=qvec, anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": 64}},
        limit=k, output_fields=["source", "text"],
    )
    return [(h.entity.get("source"), h.entity.get("text"), h.distance) for h in hits[0]]

# Helper: answer a question by RAG against `llm_client` (managed or local).
SYSTEM = (
    "You are a NRP support assistant. Answer the question using ONLY the context "
    "provided. If the context does not contain the answer, say so explicitly. "
    "Cite the source labels you used (e.g., [nrp-gpus])."
)
def ask(question, llm_client, model, max_tokens=400):
    hits = retrieve(question, k=3)
    context = "\n\n".join(f"[{src}] {txt}" for src, txt, _ in hits)
    resp = llm_client.chat.completions.create(
        model=model, temperature=0.1, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"Question: {question}\n\nContext:\n{context}"},
        ],
    )
    content = resp.choices[0].message.content or "(model produced no `content`)"
    return content, hits

QUESTION = "What happens to a pod on NRP that has no CPU or memory limits?"
print("Retrieved chunks:")
for src, _, score in retrieve(QUESTION, k=3):
    print(f"  [{src}]  score={score:.3f}")


In [ ]:
# Same question, managed gemma.
ans, hits = ask(QUESTION, client, "gemma")
print(ans)


In [ ]:
# Same question, local mistral on the session GPU.
if HAS_GPU:
    ans, hits = ask(QUESTION, local, "mistral")
    print(ans)
else:
    print("(Skipped — no GPU.)")


**What to notice.** Both backends see the same retrieved chunks (same Milvus
collection, same query embedding). Differences in the answers come purely
from the model itself:

- The managed `gemma` typically sticks to the context and cites the source.
- Smaller local models like `mistral` 7B sometimes fall back on parametric
  knowledge and add facts that aren't in the chunks — even when told not to.
  This is exactly the kind of behavior a docs-grounded assistant has to
  guard against, and trying a heavier local model (`llama3.1:70b`,
  `qwen2.5:32b`) usually makes the difference shrink.


## 5. Cleanup

The simple-RAG collection is per-user and demo-only — drop it. Stopping the
local Ollama process frees the GPU memory for other notebook cells.


In [ ]:
# Drop the per-user simple-RAG collection.
if utility.has_collection(SIMPLE_COLLECTION):
    utility.drop_collection(SIMPLE_COLLECTION)
    print(f"Dropped {SIMPLE_COLLECTION}.")
else:
    print(f"{SIMPLE_COLLECTION} already gone.")


In [ ]:
# Stop the local Ollama daemon (frees GPU memory).
import subprocess
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
print("Ollama stopped.")


## Takeaways

- The same OpenAI-compatible Python code targeted two different inference
  backends: NRP's managed `ellm.nrp-nautilus.io` and a local Ollama on this
  session's GPU. Only `base_url` changed.
- Managed LLMs are the lowest-friction path for classrooms — no token
  handoff and no GPU billed to students. A local/self-hosted model makes
  sense when you need a specific model, quantization, or runtime control.
- RAG is "embed your docs, retrieve the top-k chunks for the question, then
  prompt the LLM with 'answer from context only.'" The vector database
  (Milvus), the embedding model, and the LLM are all independently swappable.

**Next:** open the [README](README.md) for a short `opencode` example — an
agentic coding CLI pointed at this same managed LLM.
